In [1]:
from google.colab import drive; drive.mount('/content/drive')
import json

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORED = (500, 1000, 2000, 3000)   # the four checkpoints the plan scores on val

def curve(path):
    try:
        return [json.loads(l) for l in open(path)]
    except FileNotFoundError:
        return None

def show(label, rows):
    if rows is None:
        print(f"{label:<14}(no val points yet)\n"); return
    lo = min(rows, key=lambda d: d["val_loss"])
    ups = sum(1 for a, b in zip(rows, rows[1:]) if b["val_loss"] > a["val_loss"])
    print(f"{label:<14}argmin {lo['step']:>5} = {lo['val_loss']:.6f}   {ups} up")
    print(f"{'step':>8}{'val_loss':>14}{'delta':>14}")
    prev = None
    for d in rows:
        delta = "" if prev is None else f"{d['val_loss'] - prev:+.6f}"
        mark = "  *" if d["step"] in SCORED else ""
        print(f"{d['step']:>8d}{d['val_loss']:>14.6f}{delta:>14}{mark}")
        prev = d["val_loss"]

print("* = scored on the val slice\n")
for run in ["sp_i_p668", "sp_ii_p668", "sp_iii_p139"]:
    show(run, curve(f"{DRIVE}/soft_prompt/experiments/{run}/val_loss.jsonl")); print()

Mounted at /content/drive
* = scored on the val slice

sp_i_p668     argmin  2750 = 0.344604   3 up
    step      val_loss         delta
     250      0.357146              
     500      0.347585     -0.009561  *
     750      0.346419     -0.001166
    1000      0.345847     -0.000572  *
    1250      0.345595     -0.000252
    1500      0.346164     +0.000569
    1750      0.345649     -0.000515
    2000      0.345072     -0.000577  *
    2250      0.345355     +0.000283
    2500      0.345305     -0.000049
    2750      0.344604     -0.000701
    3000      0.344898     +0.000294  *

sp_ii_p668    argmin  3000 = 0.341805   1 up
    step      val_loss         delta
     250      0.343400              
     500      0.342951     -0.000449  *
     750      0.342788     -0.000163
    1000      0.342334     -0.000454  *
    1250      0.342324     -0.000011
    1500      0.342192     -0.000131
    1750      0.342121     -0.000072
    2000      0.341970     -0.000151  *
    2250      0.342

In [2]:
print(f"{'run':16s} {'first':>10s} {'last':>10s} {'argmin':>8s} {'min':>10s}")
for run in ["lora_r4", "lora_r8", "lora_r16", "lora_r8_early"]:
    rows = curve(f"{DRIVE}/lora/experiments/{run}/val_loss.jsonl")
    if rows is None:
        print(f"{run:16s} (not on Drive)"); continue
    lo = min(rows, key=lambda d: d["val_loss"])
    print(f"{run:16s} {rows[0]['val_loss']:10.6f} {rows[-1]['val_loss']:10.6f} "
          f"{lo['step']:8d} {lo['val_loss']:10.6f}")

for run in ["sp_i_p668", "sp_ii_p668", "sp_iii_p139"]:
    rows = curve(f"{DRIVE}/soft_prompt/experiments/{run}/val_loss.jsonl")
    if rows:
        lo = min(rows, key=lambda d: d["val_loss"])
        print(f"{run:16s} {rows[0]['val_loss']:10.6f} {rows[-1]['val_loss']:10.6f} "
              f"{lo['step']:8d} {lo['val_loss']:10.6f}")

run                   first       last   argmin        min
lora_r4            0.349181   0.344816     4000   0.344816
lora_r8            0.349190   0.344500     4000   0.344500
lora_r16           0.349187   0.343955     4000   0.343955
lora_r8_early      0.349180   0.347863     1000   0.347863
sp_i_p668          0.357146   0.344898     2750   0.344604
sp_ii_p668         0.343400   0.341805     3000   0.341805
sp_iii_p139        0.348581   0.343125     3000   0.343125
